# CSCI 4930: Group Project
Mark Evers  
Marlon Williams  
John Sepulvuda  
Kyle

### Install

In [1]:
# %pip install -r requirements.txt

### Imports

In [ ]:
import gdown
import numpy as np 
import os
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import precision_recall_fscore_support, classification_report
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC

### Download dataset into workspace

In [3]:
# check if the file exists
if not os.path.exists("data/ODC_CRIME_TRAFFICACCIDENTS5YR_P_7255667086186507966.csv"):
    # create the data directory if it doesn't exist
    os.makedirs("data", exist_ok=True)
    # download the file from Google Drive
    gdown.download("https://drive.google.com/uc?id=1wcvSB8gTEAzbq2ARRhjGJGyoMlP-Rby3", "data/ODC_CRIME_TRAFFICACCIDENTS5YR_P_7255667086186507966.csv", quiet=False)

### Load the data

In [4]:
keep_cols = [
    "offense_code",
    # The two below are data leaks
    # "offense_code_extension",
    # "top_traffic_accident_offense",
    "first_occurrence_date",
    "district_id",
    "precinct_id",
    "neighborhood_id",
    "bicycle_ind",
    "pedestrian_ind",
    "HARMFUL_EVENT_SEQ_1",
    "HARMFUL_EVENT_SEQ_2",
    "HARMFUL_EVENT_SEQ_3",
    "road_location",
    "ROAD_DESCRIPTION",
    "ROAD_CONTOUR",
    "ROAD_CONDITION",
    "LIGHT_CONDITION",
    "TU1_VEHICLE_TYPE",
    "TU1_TRAVEL_DIRECTION",
    "TU1_VEHICLE_MOVEMENT",
    "TU1_DRIVER_ACTION",
    "TU1_DRIVER_HUMANCONTRIBFACTOR",
    "TU1_PEDESTRIAN_ACTION",
    "TU2_VEHICLE_TYPE",
    "TU2_TRAVEL_DIRECTION",
    "TU2_VEHICLE_MOVEMENT",
    "TU2_DRIVER_ACTION",
    "TU2_DRIVER_HUMANCONTRIBFACTOR",
    "TU2_PEDESTRIAN_ACTION",
    # "geo_lon",
    # "geo_lat",
    # these are the target variables
    "SERIOUSLY_INJURED",
    "FATALITIES"
]
df = pd.read_csv("data/ODC_CRIME_TRAFFICACCIDENTS5YR_P_7255667086186507966.csv", usecols=keep_cols)
print(f"Total rows: {len(df)}")

Total rows: 282244


/tmp/ipykernel_20906/1515547268.py:38: DtypeWarning: Columns (0: LIGHT_CONDITION) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/ODC_CRIME_TRAFFICACCIDENTS5YR_P_7255667086186507966.csv", usecols=keep_cols)


In [5]:
print("Constructing high risk field...")

# Create a field where any fatality or serious injury
# is classified as "high risk".
df["high_risk"] = ((df["FATALITIES"].fillna(0) > 0) | (df["SERIOUSLY_INJURED"].fillna(0) > 0)).astype(int)
# drop the original target variables (data leak)
df.drop(columns=["SERIOUSLY_INJURED", "FATALITIES"], inplace=True)

print(f"Number of high risk accidents: {df['high_risk'].sum()}")

Constructing high risk field...
Number of high risk accidents: 6559


### Drop rows with missing values

In [6]:
# total_before = len(df)
# print(f"Total rows before dropping null values: {total_before}")

# n_before = len(df)
# df = df[(~df.SERIOUSLY_INJURED.isnull()) & (~df.FATALITIES.isnull())]
# n_after = len(df)
# print(f"Dropped {n_before - n_after} rows with null values in column SERIOUSLY_INJURED and FATALITIES.")

# n_before = len(df)
# df = df[(~df.ROAD_DESCRIPTION.isnull()) & (~df.ROAD_CONTOUR.isnull()) & (~df.ROAD_CONDITION.isnull())]
# n_after = len(df)
# print(f"Dropped {n_before - n_after} rows with null values in columns ROAD_DESCRIPTION, ROAD_CONTOUR, and ROAD_CONDITION.")

# n_before = n_after
# df = df[(~df.HARMFUL_EVENT_SEQ_1.isnull())]
# n_after = len(df)
# print(f"Dropped {n_before - n_after} rows with null values in column HARMFUL_EVENT_SEQ_1.")

# n_before = n_after
# df = df[(~df.TU1_TRAVEL_DIRECTION.isnull())]
# n_after = len(df)
# print(f"Dropped {n_before - n_after} rows with null values in column TU1_TRAVEL_DIRECTION.")

# # n_before = n_after
# # df = df[(~df.geo_lon.isnull())]
# # n_after = len(df)
# # print(f"Dropped {n_before - n_after} rows with null values in column geo_lon and geo_lat.")

# n_before = n_after
# df = df[(~df.TU1_DRIVER_ACTION.isnull())]
# n_after = len(df)
# print(f"Dropped {n_before - n_after} rows with null values in column TU1_DRIVER_ACTION.")

# n_before = n_after
# df = df[(~df.TU1_DRIVER_HUMANCONTRIBFACTOR.isnull())]
# n_after = len(df)
# print(f"Dropped {n_before - n_after} rows with null values in column TU1_DRIVER_HUMANCONTRIBFACTOR.")

# total_after = len(df)
# print(f"Total rows after dropping null values: {total_after} ({total_before - total_after} rows dropped in total).")


In [7]:
print("Trimming whitespace...")
# df["top_traffic_accident_offense"] = df["top_traffic_accident_offense"].str.strip()
df["district_id"] = df["district_id"].str.strip()

Trimming whitespace...


In [8]:
print("Filling in missing values...")

# df["geo_lon"] = df["geo_lon"].fillna(0)
# df["geo_lat"] = df["geo_lat"].fillna(0)

df["district_id"] = df["district_id"].fillna("None")
df["precinct_id"] = df["precinct_id"].fillna("None")
df["neighborhood_id"] = df["neighborhood_id"].fillna("None")
df["bicycle_ind"] = df["bicycle_ind"].fillna(0)
df["pedestrian_ind"] = df["pedestrian_ind"].fillna(0)

df["HARMFUL_EVENT_SEQ_1"] = df["HARMFUL_EVENT_SEQ_1"].fillna("None")
df["HARMFUL_EVENT_SEQ_1"] = df["HARMFUL_EVENT_SEQ_1"].replace("  ", "None")
df["HARMFUL_EVENT_SEQ_2"] = df["HARMFUL_EVENT_SEQ_2"].fillna("None")
df["HARMFUL_EVENT_SEQ_2"] = df["HARMFUL_EVENT_SEQ_2"].replace("  ", "None")
df["HARMFUL_EVENT_SEQ_3"] = df["HARMFUL_EVENT_SEQ_3"].fillna("None")
df["HARMFUL_EVENT_SEQ_3"] = df["HARMFUL_EVENT_SEQ_3"].replace("  ", "None")

df["road_location"] = df["road_location"].fillna("None")
df["road_location"] = df["road_location"].replace("  ", "None")
df["ROAD_DESCRIPTION"] = df["ROAD_DESCRIPTION"].replace("  ", "None")
df["ROAD_CONTOUR"] = df["ROAD_CONTOUR"].replace("  ", "None")
df["ROAD_CONDITION"] = df["ROAD_CONDITION"].replace("  ", "None")
df["LIGHT_CONDITION"] = df["LIGHT_CONDITION"].replace("  ", "None")
df["LIGHT_CONDITION"] = df["LIGHT_CONDITION"].fillna("None")

df["TU1_VEHICLE_TYPE"] = df["TU1_VEHICLE_TYPE"].fillna("None")
df["TU1_VEHICLE_TYPE"] = df["TU1_VEHICLE_TYPE"].replace({"  ": "None", "UNK": "None", "0": "None"})
df["TU1_TRAVEL_DIRECTION"] = df["TU1_TRAVEL_DIRECTION"].fillna("None")
df["TU1_TRAVEL_DIRECTION"] = df["TU1_TRAVEL_DIRECTION"].replace("  ", "None")
df["TU1_TRAVEL_DIRECTION"] = df["TU1_TRAVEL_DIRECTION"].replace("021", "None")
df["TU1_TRAVEL_DIRECTION"] = df["TU1_TRAVEL_DIRECTION"].replace("unk", "None")
df["TU1_TRAVEL_DIRECTION"] = df["TU1_TRAVEL_DIRECTION"].replace("91", "None")
df["TU1_VEHICLE_MOVEMENT"] = df["TU1_VEHICLE_MOVEMENT"].fillna("None")
df["TU1_VEHICLE_MOVEMENT"] = df["TU1_VEHICLE_MOVEMENT"].replace("  ", "None")
df["TU1_VEHICLE_MOVEMENT"] = df["TU1_VEHICLE_MOVEMENT"].replace("021", "None")
df["TU1_VEHICLE_MOVEMENT"] = df["TU1_VEHICLE_MOVEMENT"].replace("18", "None")
df["TU1_VEHICLE_MOVEMENT"] = df["TU1_VEHICLE_MOVEMENT"].replace("00", "None")
df["TU1_TRAVEL_DIRECTION"] = df["TU1_TRAVEL_DIRECTION"].replace("00", "None")
df["TU1_DRIVER_ACTION"] = df["TU1_DRIVER_ACTION"].fillna("None")
df["TU1_DRIVER_ACTION"] = df["TU1_DRIVER_ACTION"].replace("  ", "None")
df["TU1_DRIVER_ACTION"] = df["TU1_DRIVER_ACTION"].replace("na", "None")
df["TU1_DRIVER_ACTION"] = df["TU1_DRIVER_ACTION"].replace("00", "None")
df["TU1_DRIVER_ACTION"] = df["TU1_DRIVER_ACTION"].replace("17", "None")
df["TU1_DRIVER_HUMANCONTRIBFACTOR"] = df["TU1_DRIVER_HUMANCONTRIBFACTOR"].fillna("None")
df["TU1_DRIVER_HUMANCONTRIBFACTOR"] = df["TU1_DRIVER_HUMANCONTRIBFACTOR"].replace("  ", "None")
df["TU1_DRIVER_HUMANCONTRIBFACTOR"] = df["TU1_DRIVER_HUMANCONTRIBFACTOR"].replace(" 00", "None")
df["TU1_DRIVER_HUMANCONTRIBFACTOR"] = df["TU1_DRIVER_HUMANCONTRIBFACTOR"].replace(" 06", "None")
df["TU1_DRIVER_HUMANCONTRIBFACTOR"] = df["TU1_DRIVER_HUMANCONTRIBFACTOR"].replace("18", "None")
df["TU1_DRIVER_HUMANCONTRIBFACTOR"] = df["TU1_DRIVER_HUMANCONTRIBFACTOR"].replace("17", "None")
df["TU1_DRIVER_HUMANCONTRIBFACTOR"] = df["TU1_DRIVER_HUMANCONTRIBFACTOR"].replace("NO APPARENT", "No Apparent Contributing Factor")
df["TU1_PEDESTRIAN_ACTION"] = df["TU1_PEDESTRIAN_ACTION"].fillna("None")
df["TU1_PEDESTRIAN_ACTION"] = df["TU1_PEDESTRIAN_ACTION"].replace("  ", "None")
df["TU1_PEDESTRIAN_ACTION"] = df["TU1_PEDESTRIAN_ACTION"].replace("00", "None")
df["TU1_PEDESTRIAN_ACTION"] = df["TU1_PEDESTRIAN_ACTION"].replace("/", "None")
df["TU1_PEDESTRIAN_ACTION"] = df["TU1_PEDESTRIAN_ACTION"].replace("-", "None")
df["TU1_PEDESTRIAN_ACTION"] = df["TU1_PEDESTRIAN_ACTION"].replace("--", "None")

df["TU2_VEHICLE_TYPE"] = df["TU2_VEHICLE_TYPE"].fillna("None")
df["TU2_VEHICLE_TYPE"] = df["TU2_VEHICLE_TYPE"].replace("  ", "None")
df["TU2_TRAVEL_DIRECTION"] = df["TU2_TRAVEL_DIRECTION"].fillna("None")
df["TU2_TRAVEL_DIRECTION"] = df["TU2_TRAVEL_DIRECTION"].replace("  ", "None")
df["TU2_TRAVEL_DIRECTION"] = df["TU2_TRAVEL_DIRECTION"].replace("05", "None")
df["TU2_TRAVEL_DIRECTION"] = df["TU2_TRAVEL_DIRECTION"].replace("07", "None")
df["TU2_TRAVEL_DIRECTION"] = df["TU2_TRAVEL_DIRECTION"].replace("01", "None")
df["TU2_TRAVEL_DIRECTION"] = df["TU2_TRAVEL_DIRECTION"].replace("03", "None")
df["TU2_TRAVEL_DIRECTION"] = df["TU2_TRAVEL_DIRECTION"].replace("00", "None")
df["TU2_TRAVEL_DIRECTION"] = df["TU2_TRAVEL_DIRECTION"].replace("02", "None")
df["TU2_VEHICLE_MOVEMENT"] = df["TU2_VEHICLE_MOVEMENT"].fillna("None")
df["TU2_VEHICLE_MOVEMENT"] = df["TU2_VEHICLE_MOVEMENT"].replace("  ", "None")
df["TU2_VEHICLE_MOVEMENT"] = df["TU2_VEHICLE_MOVEMENT"].replace("00", "None")
df["TU2_VEHICLE_MOVEMENT"] = df["TU2_VEHICLE_MOVEMENT"].replace("0", "None")
df["TU2_DRIVER_ACTION"] = df["TU2_DRIVER_ACTION"].fillna("None")
df["TU2_DRIVER_ACTION"] = df["TU2_DRIVER_ACTION"].replace("  ", "None")
df["TU2_DRIVER_ACTION"] = df["TU2_DRIVER_ACTION"].replace("00", "None")
df["TU2_DRIVER_ACTION"] = df["TU2_DRIVER_ACTION"].replace("0", "None")
df["TU2_DRIVER_ACTION"] = df["TU2_DRIVER_ACTION"].replace("non", "None")
df["TU2_DRIVER_ACTION"] = df["TU2_DRIVER_ACTION"].replace("none", "None")
df["TU2_DRIVER_HUMANCONTRIBFACTOR"] = df["TU2_DRIVER_HUMANCONTRIBFACTOR"].fillna("None")
df["TU2_DRIVER_HUMANCONTRIBFACTOR"] = df["TU2_DRIVER_HUMANCONTRIBFACTOR"].replace("  ", "None")
df["TU2_PEDESTRIAN_ACTION"] = df["TU2_PEDESTRIAN_ACTION"].fillna("None")
df["TU2_PEDESTRIAN_ACTION"] = df["TU2_PEDESTRIAN_ACTION"].replace("  ", "None")
df["TU2_PEDESTRIAN_ACTION"] = df["TU2_PEDESTRIAN_ACTION"].replace("00", "None")
df["TU2_PEDESTRIAN_ACTION"] = df["TU2_PEDESTRIAN_ACTION"].replace("0", "None")
df["TU2_PEDESTRIAN_ACTION"] = df["TU2_PEDESTRIAN_ACTION"].replace("/", "None")
df["TU2_PEDESTRIAN_ACTION"] = df["TU2_PEDESTRIAN_ACTION"].replace("--", "None")
df["TU2_PEDESTRIAN_ACTION"] = df["TU2_PEDESTRIAN_ACTION"].replace("-", "None")

# convert the pedestrian and bicyclist indicators to binary fields
# these might be a data leak
df["pedestrian_ind"] = (df["pedestrian_ind"].fillna(0) > 0).astype(int)
df["bicycle_ind"] = (df["bicycle_ind"].fillna(0) > 0).astype(int)


Filling in missing values...


In [9]:
# It is necessary to separate the time of the incident 
# into multiple categories.
print("Extracting time features...")

df["first_occurrence_date"] = pd.to_datetime(
    df["first_occurrence_date"], errors="coerce"
)

df["hour"] = df["first_occurrence_date"].dt.hour
df["day_of_week"] = df["first_occurrence_date"].dt.dayofweek   # 0=Mon 6=Sun
df["month"] = df["first_occurrence_date"].dt.month

# We no longer need the raw datetime column
df.drop(columns=["first_occurrence_date"], inplace=True)

Extracting time features...


In [10]:
df.to_csv("data/processed_crime_traffic_accidents.csv", index=False)

In [11]:
# force all columns to be strings (except the target variable)
for col in df.columns:
    if col != "high_risk":
        df[col] = df[col].astype(str)

In [12]:
X = df.drop(columns=["high_risk"])
y = df["high_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=1234,
    stratify=y,      # keep class balance in both splits
)

In [13]:
# all the data is categorical, one-hot encode everything

encoder = OneHotEncoder(handle_unknown="ignore")
X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)
X_train_encoded.shape

(225795, 981)

In [ ]:
print("Performing Random Forest Grid Search...")

# make sure the classes are balanced
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=1234)

# build the grid search
param_grid = {
    "min_samples_leaf": [i for i in range(5, 22, 2)],
    "max_features": [i for i in range(10, 101, 10)],
    "n_estimators": [i for i in range(300, 1001, 100)]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(class_weight='balanced', oob_score=True, random_state=1234),
    param_grid=param_grid,
    cv=skf,
    n_jobs=-1,
    scoring='f1', # Since our classes are imbalanced, don't use 'accuracy'
    refit=True
)

# find the best parameters
grid_search.fit(X_train_encoded, y_train)

Performing Grid Search...


/home/mark/school/2026 Spring/CSCI 4930 - Machine Learning/project/.venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [ ]:
print("Best Random Forest CV Score:", grid_search.best_score_)
print("Best Random Forest Parameters:", grid_search.best_params_)

# save the winning model to a variable
rf_model = grid_search.best_estimator_
rf_model.fit(X_train_encoded, y_train)
y_pred_test = rf_model.predict(X_test)

# print the results
print(classification_report(y_test, y_pred_test, digits=4))
precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred_test)

# pickle the model
os.makedirs("pickle_jar", exist_ok=True)
with open(f"./pickle_jar/rf_model_{f1:.4f}.pkl", "wb") as f:
    pickle.dump(rf_model, f) 

In [ ]:
print("Performing AdaBoost Grid Search...")

# make sure the classes are balanced
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=1234)

# build the grid search
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 1.0],
    'estimator__max_depth': [1, 2] # Accessing the base tree parameters
}

ada_grid_search = GridSearchCV(
    estimator=AdaBoostClassifier(estimator=DecisionTreeClassifier(class_weight='balanced')),
    param_grid=param_grid,
    scoring='f1', # Crucial for our fatality predictions
    cv=3,
    n_jobs=-1,
    refit=True,
    verbose=1
)

# find the best parameters
ada_grid_search.fit(X_train_encoded, y_train)

# print some of the results:
print("Best AdaBoost CV F1 Score:", ada_grid_search.best_score_)
print("Best AdaBoost Params:", ada_grid_search.best_params_)

# save the winning model to a variable
ada_model = ada_grid_search.best_estimator_
y_pred_test = ada_model.predict(X_test)

# print the results
print(classification_report(y_test, y_pred_test, digits=4))
precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred_test)

# pickle the model
os.makedirs("pickle_jar", exist_ok=True)
with open(f"./pickle_jar/ada_model_{f1:.4f}.pkl", "wb") as f:
    pickle.dump(ada_model, f) 

In [ ]:
# make sure the classes are balanced
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)


# build the grid search
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'loss': ['hinge', 'squared_hinge']
}

svm = LinearSVC(class_weight='balanced', dual=False, random_state=42, max_iter=2000)

svm_grid = GridSearchCV(
    estimator=svm,
    param_grid=param_grid,
    cv=skf,
    scoring='f1', # Still focusing on that balance of Precision and Recall
    n_jobs=-1,
    refit=True,
    verbose=1
)

# find best parameters with grid search
svm_grid.fit(X_train, y_train)

# print some of the results:
print("Best SVM CV F1 Score:", svm_grid.best_score_)
print("Best SVM Params:", svm_grid.best_params_)

# save the winning model to a variable
best_svm = svm_grid.best_estimator_
y_pred_test = best_svm.predict(X_test)

# print the results
print(classification_report(y_test, y_pred_test, digits=4))
precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred_test)

# pickle the model
os.makedirs("pickle_jar", exist_ok=True)
with open(f"./pickle_jar/svm_model_{f1:.4f}.pkl", "wb") as f:
    pickle.dump(best_svm, f) 